In [1]:
import sys
sys.path.append('/home/quant/qlib')
import pickle
import xqbutils as xqb
import pandas as pd
import os
import shutil
import subprocess
import logging
# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

threshold = 0.9
def get_stockinfo_wd():
    print('开始处理日行情数据')
    df = xqb.get_oracal_data(xqb.PRICES_CONFIG, 'ASHAREEODPRICES')
    df.rename(columns = {'S_INFO_WINDCODE':'STOCKCODE','TRADE_DT':'date','S_DQ_ADJOPEN':'OPEN','S_DQ_ADJHIGH':'HIGH','S_DQ_ADJLOW':'LOW','S_DQ_ADJCLOSE':'CLOSE','S_DQ_VOLUME':'VOLUME','S_DQ_AVGPRICE':'VWAP','S_DQ_ADJFACTOR':'FACTOR'}, inplace = True)
    df = df[['STOCKCODE','date','OPEN','HIGH','LOW','CLOSE','VOLUME','VWAP','FACTOR']]
    df['date'] = pd.to_datetime(df['date'])
    null_proportions = df.isnull().sum() / len(df)
    columns_to_drop = null_proportions[null_proportions > threshold].index
    df = df.drop(columns = columns_to_drop)
    df.set_index('date',inplace=True)
    df = df.sort_index()
    df = df['2014-01-01':]
    df.reset_index(drop=False, inplace=True)
    grouped = df.groupby('STOCKCODE')
    i = 0
    for name, group in grouped:
        group = group.drop(columns =['STOCKCODE'])
        group = group.sort_values(by='date', ascending=True)
        group.to_csv(f'/home/quant/qlib/stockinfo/{name}.csv',index=False)
        i += 1
        print(f'{i}:{name}')

def get_indexdaily():
    df = xqb.get_oracal_data(xqb.FACTOR_CONFIG,'JY_QY_INDEXDATA_CS')
    df = df.drop(columns=['ID','CREATETIME'])
    df.rename(columns = {'TRADEDATE':'date','TOPEN':'OPEN','THIGH':'HIGH','TLOW':'LOW','TCLOSE':'CLOSE'}, inplace = True)
    df['date'] = pd.to_datetime(df['date'])
    null_proportions = df.isnull().sum() / len(df)
    columns_to_drop = null_proportions[null_proportions > threshold].index
    df = df.drop(columns = columns_to_drop)
    df.set_index('date',inplace=True)
    df = df.sort_index()
    df = df['2014-01-01':]
    df.reset_index(drop=False, inplace=True)
    grouped = df.groupby('INDEXCODE')
    for name, group in grouped:
        group = group.drop(columns =['INDEXCODE'])
        group = group.sort_values(by='date', ascending=True)
        group.to_csv(f'/home/quant/qlib/indexinfo/{name}.ZS.csv',index=False)

# 公司股票池处理
def get_company_stock():
    column_names = ['SYMBOL', 'starttime', 'endtime']
    df_stock = pd.read_table("/home/quant/qlib_data/cn_data/instruments/all.txt", names=column_names)
    column_names = ['SYMBOL']
    df_company = pd.read_csv("/home/quant/qlib_data/section_codes.csv", names=column_names,dtype = str)
    result = df_stock[df_stock['SYMBOL'].str[:6].isin(df_company['SYMBOL'])]
    result.to_csv("/home/quant/qlib_data/cn_data/instruments/company.txt", sep='\t', index=False, header=False)


def get_csi300_stock():
    column_names = ['SYMBOL', 'starttime', 'endtime']
    df_stock = pd.read_table("/home/quant/qlib_data/cn_data/instruments/all.txt", names=column_names)
    df_company = pd.read_csv("/home/quant/qlib_data/csi300.csv", names=column_names)
    result = df_stock[df_stock['SYMBOL'].isin(df_company['SYMBOL'])]
    result.to_csv("/home/quant/qlib_data/cn_data/instruments/csi300.txt", sep='\t', index=False, header=False)

def save_data_company():
    """
    Saves the dataset to a pickle file.
    """
    import qlib
    from qlib.constant import REG_CN
    from qlib.utils import init_instance_by_config
    from datetime import datetime

    from qlib.data import D # 基础行情数据服务的对象
    provider_uri = "/home/quant/qlib_data/cn_data"

    #数据初始化，
    qlib.init(provider_uri=provider_uri, region=REG_CN) # 初始化

    stockpool =  D.instruments(market='company')

    end_time = datetime.now().strftime("%Y-%m-%d")
    print(f"今天的日期是: {end_time}")

    data_handler_config = {
        "start_time": "2008-01-01",
        "end_time": end_time,
        "fit_start_time": "2008-01-01",
        "fit_end_time": "2021-12-31",
        "instruments": stockpool,
        "infer_processors":[
            {"class": "RobustZScoreNorm",
              "kwargs":{
                  "fields_group": "feature",
                  "clip_outlier": "true"
              }
             },
            {"class": "Fillna",
              "kwargs":{
                  "fields_group": "feature"
              }
             },
        ],
        "label": [["Ref($close, -2) / Ref($close, -1) - 1"], ["LABEL0"]]
    }

    task = {
        "dataset": {  #　因子数据集参数配置
            # 数据集类，是Dataset with Data(H)andler的缩写，即带数据处理器的数据集
            "class": "TSDatasetH",
            "module_path": "qlib.data.dataset",
            "kwargs": {
                "handler": { # 数据集使用的数据处理器配置
                    "class": "AlphaTa", # 数据处理器类，继承自DataHandlerLP
                    "module_path": "qlib.contrib.data.handler_xie", # 数据处理器类所在模块
                    "kwargs": data_handler_config, # 数据处理器参数配置
                },
                "segments": {
                    "train": ("2008-01-01", "2021-12-31"),
                    "valid": ("2022-01-01", "2023-12-31"),
                    "test": ("2024-01-01", end_time),
                },
                "step_len": 20,  # 数据集步长
            },
        },
    }

    dataset_ts = init_instance_by_config(task["dataset"]) # 类型DatasetH
    dataset_ts.config(dump_all=True, recursive=True)
    dataset_ts.to_pickle(path="dataset_ts.pkl", dump_all=True)

    task = {
        "dataset": {  #　因子数据集参数配置
            # 数据集类，是Dataset with Data(H)andler的缩写，即带数据处理器的数据集
            "class": "DatasetH",
            "module_path": "qlib.data.dataset",
            "kwargs": {
                "handler": { # 数据集使用的数据处理器配置
                    "class": "AlphaTa", # 数据处理器类，继承自DataHandlerLP
                    "module_path": "qlib.contrib.data.handler_xie", # 数据处理器类所在模块
                    "kwargs": data_handler_config, # 数据处理器参数配置
                },
                "segments": {
                    "train": ("2008-01-01", "2021-12-31"),
                    "valid": ("2022-01-01", "2023-12-31"),
                    "test": ("2024-01-01", end_time),
                },
            },
        },
    }
    # 实例化数据集，从基础行情数据计算出的包含所有特征（因子）和标签值的数据集。
    dataset = init_instance_by_config(task["dataset"]) # 类型DatasetH
    dataset.config(dump_all=True, recursive=True)
    dataset.to_pickle(path="dataset.pkl", dump_all=True)


def save_data_all():
    """
    Saves the dataset to a pickle file.
    """
    import qlib
    from qlib.constant import REG_CN
    from qlib.utils import init_instance_by_config
    from datetime import datetime

    from qlib.data import D # 基础行情数据服务的对象
    provider_uri = "/home/quant/qlib_data/cn_data"

    #数据初始化，
    qlib.init(provider_uri=provider_uri, region=REG_CN) # 初始化

    stockpool =  D.instruments(market='all')

    end_time = datetime.now().strftime("%Y-%m-%d")
    print(f"今天的日期是: {end_time}")

    data_handler_config = {
        "start_time": "2014-01-01",
        "end_time": end_time,
        "fit_start_time": "2014-01-01",
        "fit_end_time": "2021-12-31",
        "instruments": stockpool,
        "infer_processors":[
            {"class": "RobustZScoreNorm",
              "kwargs":{
                  "fields_group": "feature",
                  "clip_outlier": "true"
              }
             },
            {"class": "Fillna",
              "kwargs":{
                  "fields_group": "feature"
              }
             },
        ],
        "label": [["Ref($close, -2) / Ref($close, -1) - 1"], ["LABEL0"]]
    }

    # task = {
    #     "dataset": {  #　因子数据集参数配置
    #         # 数据集类，是Dataset with Data(H)andler的缩写，即带数据处理器的数据集
    #         "class": "TSDatasetH",
    #         "module_path": "qlib.data.dataset",
    #         "kwargs": {
    #             "handler": { # 数据集使用的数据处理器配置
    #                 "class": "AlphaTa", # 数据处理器类，继承自DataHandlerLP
    #                 "module_path": "qlib.contrib.data.handler_xie", # 数据处理器类所在模块
    #                 "kwargs": data_handler_config, # 数据处理器参数配置
    #             },
    #             "segments": {
    #                 "train": ("2014-01-01", "2021-12-31"),
    #                 "valid": ("2022-01-01", "2023-12-31"),
    #                 "test": ("2024-01-01", end_time),
    #             },
    #             "step_len": 20,  # 数据集步长
    #         },
    #     },
    # }
    #
    # dataset_ts = init_instance_by_config(task["dataset"]) # 类型DatasetH
    # dataset_ts.config(dump_all=True, recursive=True)
    # dataset_ts.to_pickle(path="dataset_ts_all.pkl", dump_all=True)

    task = {
        "dataset": {  #　因子数据集参数配置
            # 数据集类，是Dataset with Data(H)andler的缩写，即带数据处理器的数据集
            "class": "DatasetH",
            "module_path": "qlib.data.dataset",
            "kwargs": {
                "handler": { # 数据集使用的数据处理器配置
                    "class": "AlphaTa", # 数据处理器类，继承自DataHandlerLP
                    "module_path": "qlib.contrib.data.handler_xie", # 数据处理器类所在模块
                    "kwargs": data_handler_config, # 数据处理器参数配置
                },
                "segments": {
                    "train": ("2014-01-01", "2021-12-31"),
                    "valid": ("2022-01-01", "2023-12-31"),
                    "test": ("2024-01-01", end_time),
                },
            },
        },
    }
    # 实例化数据集，从基础行情数据计算出的包含所有特征（因子）和标签值的数据集。
    dataset = init_instance_by_config(task["dataset"]) # 类型DatasetH
    dataset.config(dump_all=True, recursive=True)
    dataset.to_pickle(path="/home/quant/qlib/dataset_all.pkl", dump_all=True)


def load_data(is_ts):
    """
    Loads the dataset from the pickle file.
    """
    if is_ts:
        with open("/home/quant/qlib/dataset_ts.pkl", "rb") as file_dataset:
            dataset = pickle.load(file_dataset)
        return dataset
    else:
        with open("/home/quant/qlib/dataset.pkl", "rb") as file_dataset:
            dataset = pickle.load(file_dataset)
        return dataset

def load_data_all(is_ts):
    """
    Loads the dataset from the pickle file.
    """
    if is_ts:
        with open("/home/quant/qlib/dataset_ts_all.pkl", "rb") as file_dataset:
            dataset = pickle.load(file_dataset)
        return dataset
    else:
        with open("/home/quant/qlib/dataset_all.pkl", "rb") as file_dataset:
            dataset = pickle.load(file_dataset)
        return dataset

def load_model(model_name):
    """
    Loads the model from the pickle file.
    """
    model_name = "/home/quant/qlib/saved_models/" + model_name + ".pkl"
    with open(model_name, "rb") as file_model:
        model = pickle.load(file_model)
    return model

def get_pred_scores(modelname, is_ts):
    """
    Gets the predictions and scores from the model.
    """
    model = load_model(modelname)
    if "_all" in modelname:
        dataset = load_data_all(is_ts)
    else:
        dataset = load_data(is_ts)
    pred_score = model.predict(dataset)
    pred_score = pred_score.rename('score')
    pred_score = pred_score.to_frame('score')
    last_date = pred_score.index.get_level_values('datetime').max()

    # 提取最后一天的所有股票数据
    last_day_scores = pred_score.xs(last_date, level='datetime')
    last_day_scores = last_day_scores.reset_index()

    # # 按 score 降序排序并取前100
    # top_100 = last_day_scores.sort_values('score', ascending=False).head(100)
    #
    # # 重置索引（可选，如果希望 instrument 变成列）
    # top_100 = top_100.reset_index()

    return last_day_scores

def get_pred_scores_mix():
    """
    Gets the predictions and scores from the model.
    """
    w_gru = 0.3
    w_moe = 0.3
    w_lgb = 0.4

    dataset_ts = load_data(True)
    dataset = load_data(False)

    model = load_model("gru_company")
    pred_score_gru = model.predict(dataset_ts)
    pred_score_gru = pred_score_gru.rename('score')
    pred_score_gru = pred_score_gru.to_frame('score')
    last_date = pred_score_gru.index.get_level_values('datetime').max()
    last_day_scores_gru = pred_score_gru.xs(last_date, level='datetime')
    last_day_scores_gru = last_day_scores_gru.reset_index()

    model = load_model("moe_company")
    pred_score_moe = model.predict(dataset_ts)
    pred_score_moe = pred_score_moe.rename('score')
    pred_score_moe = pred_score_moe.to_frame('score')
    last_date = pred_score_moe.index.get_level_values('datetime').max()
    last_day_scores_moe = pred_score_moe.xs(last_date, level='datetime')
    last_day_scores_moe = last_day_scores_moe.reset_index()

    model = load_model("lgb_company")
    pred_score_lgb = model.predict(dataset)
    pred_score_lgb = pred_score_lgb.rename('score')
    pred_score_lgb = pred_score_lgb.to_frame('score')
    last_date = pred_score_lgb.index.get_level_values('datetime').max()
    last_day_scores_lgb = pred_score_lgb.xs(last_date, level='datetime')
    last_day_scores_lgb = last_day_scores_lgb.reset_index()

    # 合并三个 DataFrame
    merged_df = pd.merge(
        last_day_scores_gru,
        last_day_scores_moe,
        on="instrument",
        suffixes=("_gru", "_moe")
    )
    merged_df = pd.merge(
        merged_df,
        last_day_scores_lgb,
        on="instrument"
    )

    # 计算加权平均 score
    merged_df["score"] = (
        merged_df["score_gru"] * w_gru +
        merged_df["score_moe"] * w_moe +
        merged_df["score"] * w_lgb  # 注意：这里的 score 是 lgb 的列
    )

    # 只保留 instrument 和最终 score
    final_scores = merged_df[["instrument", "score"]]

    return final_scores

def delete_files_in_folder(folder_path):
    """
    删除指定文件夹下的所有文件和子文件夹

    参数:
        folder_path (str): 要清空的文件夹路径
    """
    try:
        # 检查文件夹是否存在
        if not os.path.exists(folder_path):
            print(f"文件夹 {folder_path} 不存在")
            return

        # 遍历文件夹
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)

            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)  # 删除文件或链接
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)  # 删除子文件夹及其内容
            except Exception as e:
                print(f"删除 {file_path} 失败. 原因: {e}")

        print(f"已成功清空文件夹 {folder_path}")
    except Exception as e:
        print(f"处理文件夹 {folder_path} 时出错. 原因: {e}")

def clean_data():
    delete_files_in_folder('/home/quant/qlib/stockinfo')
    delete_files_in_folder('/home/quant/qlib/indexinfo')
    delete_files_in_folder('/home/quant/qlib_data/cn_data')


def run_dump_bin_script_index():
    """
    执行指定的Python脚本命令
    """
    command = [
        sys.executable,  # 使用当前Python解释器
        "/home/quant/qlib/scripts/dump_bin.py",
        "dump_all",
        "--csv_path", "/home/quant/qlib/indexinfo/",
        "--qlib_dir", "/home/quant/qlib_data/cn_data"
    ]

    try:
        # 执行命令并捕获输出
        result = subprocess.run(
            command,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        print(result.stdout)

        if result.stderr:
            print("错误信息：")
            print(result.stderr)

        return True
    except subprocess.CalledProcessError as e:
        print(f"命令执行失败，返回码：{e.returncode}")
        print("错误输出：")
        print(e.stderr)
        return False

def run_dump_bin_script_stock():
    """
    执行指定的Python脚本命令
    """
    command = [
        sys.executable,  # 使用当前Python解释器
        "/home/quant/qlib/scripts/dump_bin.py",
        "dump_all",
        "--csv_path", "/home/quant/qlib/stockinfo/",
        "--qlib_dir", "/home/quant/qlib_data/cn_data"
    ]

    try:
        # 执行命令并捕获输出
        result = subprocess.run(
            command,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        print(result.stdout)

        if result.stderr:
            print("错误信息：")
            print(result.stderr)

        return True
    except subprocess.CalledProcessError as e:
        print(f"命令执行失败，返回码：{e.returncode}")
        print("错误输出：")
        print(e.stderr)
        return False

def get_orders(holding_stocks,last_day_scores):
    sell_stocks =[]
    buy_stocks = []
    #计算当前持仓不在预测股票列表中的股票（主要是不在公司股票池的情况）

    # 计算2个股票的交集
    common_elements = list(set(holding_stocks) & set(last_day_scores['instrument'].tolist()))
    # 计算不在股票池的股票
    difference = list(set(holding_stocks) - set(common_elements))
    # 存在不在股票池的股票
    difference_length = len(difference)
    # append使用错误.....
    sell_stocks = sell_stocks.append( difference)
    if difference_length > 4:
        sell_stocks = sell_stocks[:4]
    else:
        valid_codes = [code for code in holding_stocks if code in last_day_scores['instrument'].values]
        sorted_df = last_day_scores[last_day_scores['instrument'].isin(valid_codes)].sort_values(
            by='score',
            ascending=True
        ).reset_index(drop=True)
        sell_stocks = sell_stocks.append(sorted_df['instrument'][:4 - difference_length].tolist())

    top_100 = last_day_scores.sort_values('score', ascending=False).head(100)

    # 重置索引（可选，如果希望 instrument 变成列）
    top_100 = top_100.reset_index()
    filtered_top_100 = top_100[~top_100['instrument'].isin(sorted_df['instrument'])]
    buy_stocks = buy_stocks.append(filtered_top_100['instrument'].tolist()[:4])
    return sell_stocks,buy_stocks

def get_orders_n(holding_stocks, last_day_scores):
    # 初始化买卖股票列表
    sell_stocks = []
    buy_stocks = []

    # ===== 卖出逻辑 =====
    # 当前持有的股票中，哪些在预测列表中（即 last_day_scores 中的 instrument）
    holding_set = set(holding_stocks)
    pred_instruments = set(last_day_scores['instrument'].tolist())

    # 当前持仓中，不在预测列表中的股票 -> 可能要卖出
    not_in_pred = list(holding_set - pred_instruments)
    not_in_pred_length = len(not_in_pred)

    # 如果不在预测池中的持仓股票超过4个，只卖前4个（这里直接取前4个或者全部？目前取全部然后截断）
    if not_in_pred_length > 4:
        sell_stocks = not_in_pred[:4]  # 只卖前4个
    else:
        sell_stocks = not_in_pred[:]  # 先卖所有不在预测池中的持仓
        # 如果还有剩余卖出名额，从【预测池中的持仓】里，得分低的股票中补充卖出
        remaining_to_sell = 4 - not_in_pred_length

        if remaining_to_sell > 0:
            # 当前持仓中，且在预测池中的股票
            valid_codes = [code for code in holding_stocks if code in pred_instruments]
            # 找出这些股票在 last_day_scores 中的得分，并按得分升序（得分低的可能表现不好）
            sorted_holding_in_pred = last_day_scores[
                last_day_scores['instrument'].isin(valid_codes)
            ].sort_values(by='score', ascending=True).reset_index(drop=True)

            # 补充卖出得分最低的 remaining_to_sell 只
            sell_stocks.extend(sorted_holding_in_pred['instrument'].head(remaining_to_sell).tolist())

    # ===== 买入逻辑 =====
    # 取得分最高的前 100 只股票
    top_100 = last_day_scores.sort_values(by='score', ascending=False).head(100)
    top_100_instruments = set(top_100['instrument'].tolist())

    # 假设我们不想重复买已经持仓的股票（或者根据某些策略排除）
    # 这里简单起见：从 top_100 中排除当前已经持仓的股票
    available_to_buy = top_100[~top_100['instrument'].isin(holding_stocks)].reset_index(drop=True)

    # 买入得分最高的前 4 只（未被持仓的）
    buy_stocks = available_to_buy['instrument'].head(4).tolist()

    return sell_stocks, buy_stocks

# 获取300指数当前列表
def get_csi300():
    column_names = ['SYMBOL', 'starttime', 'endtime']
    df_csi300 = pd.read_csv("/home/quant/qlib_data/csi300.csv", names=column_names)
    return df_csi300['SYMBOL'].tolist()

# 计算持仓中csi300指数股的个数
def get_csi300_num(holding_stocks):
    return len(set(holding_stocks) & set(get_csi300()))

# 计算需要卖出的指数股个数和买入的指数股个数,指数为60%，30个
def get_csi300_orders_num(holding_stocks,target = 30):
    csi300_num = get_csi300_num(holding_stocks)
    diff = target - csi300_num
    if diff > 0:
        return diff, 0
    elif diff < 0:
        return 0, -diff
    else:
        return 3, 3

# 计算指数外股票买卖数量
def get_out_csi300_orders_num(holding_stocks,target = 50):
    # 计算需要买卖的指数股数量
    csi300_buy_num, csi300_sell_num = get_csi300_orders_num(holding_stocks)
    # hold_num = len(holding_stocks) - target
    #持仓超过50只
    #持仓低于50只
    #持仓为50只，当前只处理了刚好50只
    out_buy_num = 0
    out_sell_num = 0
    if csi300_buy_num == csi300_sell_num:
        out_buy_num = 2
        out_sell_num = 2
    elif csi300_buy_num > csi300_sell_num:
        out_buy_num = 0
        out_sell_num = csi300_buy_num
    else:
        out_buy_num = csi300_sell_num
        out_sell_num = 0
    return out_buy_num, out_sell_num

"""
当前持仓中，指数的排序，卖出倒数三只指数股
计算所有指数排序，计算持仓外指数排名，买入前3名
当前只处理公司股票池
"""
def proces_csi300_stock(holding_stocks,last_day_scores):

    csi300_buy_num, csi300_sell_num = get_csi300_orders_num(holding_stocks)
    out_buy_num, out_sell_num = get_out_csi300_orders_num(holding_stocks)

    csi300 = get_csi300()

    holding_set = set(holding_stocks)
    # 持仓中的指数股
    in_300 = list(holding_set & set(csi300))
    # 未持仓的指数股
    left_300 = list(set(csi300) - set(in_300))

    # 持仓中的非指数股
    out_300 = list(holding_set - set(in_300))
    # 未持仓的非指数股,所有股票-300 - 已持有的非300
    left_out_300 = list(set(last_day_scores['instrument'].to_list()) - set(csi300) - set(out_300))


    #持仓的指数股排序，由低到高
    sorted_holding_in_csi300 = last_day_scores[
                last_day_scores['instrument'].isin(in_300)
            ].sort_values(by='score', ascending=True).reset_index(drop=True)

    #持仓中非指数股的排序,，由低到高
    sorted_holding_out_csi300 = last_day_scores[
                last_day_scores['instrument'].isin(out_300)
            ].sort_values(by='score', ascending=True).reset_index(drop=True)

    # 未持仓的指数股排序，从高到底
    sorted_left_300 = last_day_scores[
                last_day_scores['instrument'].isin(left_300)
            ].sort_values(by='score', ascending=False).reset_index(drop=True)

    # 未持仓的非指数股排序，从高到底
    sorted_left_out_300 = last_day_scores[
                last_day_scores['instrument'].isin(left_out_300)
            ].sort_values(by='score', ascending=False).reset_index(drop=True)

def get_trade_stock(hold_stock_list,perd_scord):
    # 输入：持仓的股票列表，预测的得分列表，账户余额
    # 输出：卖出的股票列表，买入的股票列表,sell_list,buy_list
    # 约束：1、买卖数量限制 2、优先卖出不是股票池的股票（perd_scord中的股票即为股票池）3、指数股数量约束4、行业约束（行业占比？条件是什么）
    # 5、去掉连续涨停的股票（预测前n天，n>=2，不同的交易所涨跌幅不一样,或者最近2天涨幅超过19.5%，或者最近3天涨幅超过29%的)
    # 6、持有时间的约束（至少持有N天）7、市值的约束（市值低于50亿的股票不能超过N只）

    # 股票池本身约束
    # 1、ST的约束
    # 2、新股约束


    """
    1、约束
    买卖数量限制/优先卖出不是股票池的股票/指数股数量约束/行业约束/持有时间的约束(至少持有N天)/市值的约束(市值低于M亿的股票不能超过N只)/连续涨跌停约束

    2、回测
    输入：预测分数/账户余额(回测一般为一个亿)/约束（多条件约束如何传入）
    输出：回测相关指标

    3、预测
    输入：现有持仓/预测分数/账户余额/约束（多条件约束如何传入）
    输出：卖出列表，买入列表
    """


    pass



In [3]:
# 初始化环境，每天只需执行一次
# 删除历史数据、获取最新行情数据、整理成qlib格式文件、生成数据文件
clean_data()
get_indexdaily()
get_stockinfo_wd()
run_dump_bin_script_index()
run_dump_bin_script_stock()
get_company_stock()
get_csi300_stock()
save_data_company()
# save_data_all()

已成功清空文件夹 /home/quant/qlib/stockinfo
已成功清空文件夹 /home/quant/qlib/indexinfo
已成功清空文件夹 /home/quant/qlib_data/cn_data
开始处理日行情数据
1:000001.SZ
2:000002.SZ
3:000004.SZ
4:000005.SZ
5:000006.SZ
6:000007.SZ
7:000008.SZ
8:000009.SZ
9:000010.SZ
10:000011.SZ
11:000012.SZ
12:000014.SZ
13:000016.SZ
14:000017.SZ
15:000018.SZ
16:000019.SZ
17:000020.SZ
18:000021.SZ
19:000023.SZ
20:000024.SZ
21:000025.SZ
22:000026.SZ
23:000027.SZ
24:000028.SZ
25:000029.SZ
26:000030.SZ
27:000031.SZ
28:000032.SZ
29:000033.SZ
30:000034.SZ
31:000035.SZ
32:000036.SZ
33:000037.SZ
34:000038.SZ
35:000039.SZ
36:000040.SZ
37:000042.SZ
38:000045.SZ
39:000046.SZ
40:000048.SZ
41:000049.SZ
42:000050.SZ
43:000055.SZ
44:000056.SZ
45:000058.SZ
46:000059.SZ
47:000060.SZ
48:000061.SZ
49:000062.SZ
50:000063.SZ
51:000065.SZ
52:000066.SZ
53:000068.SZ
54:000069.SZ
55:000070.SZ
56:000078.SZ
57:000088.SZ
58:000089.SZ
59:000090.SZ
60:000096.SZ
61:000099.SZ
62:000100.SZ
63:000150.SZ
64:000151.SZ
65:000153.SZ
66:000155.SZ
67:000156.SZ
68:000157.SZ
69:0

[3669930:MainThread](2025-09-01 17:05:22,398) INFO - qlib.Initialization - [config.py:451] - default_conf: client.
[3669930:MainThread](2025-09-01 17:05:23,552) WARNING - qlib.OpsWrapper - [ops.py:1900] - The custom operator [EMA] will override the qlib default definition
[3669930:MainThread](2025-09-01 17:05:23,553) WARNING - qlib.OpsWrapper - [ops.py:1900] - The custom operator [WMA] will override the qlib default definition
[3669930:MainThread](2025-09-01 17:05:23,555) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[3669930:MainThread](2025-09-01 17:05:23,557) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/home/quant/qlib_data/cn_data')}


今天的日期是: 2025-09-01


[3669930:MainThread](2025-09-01 17:06:06,564) INFO - qlib.timer - [log.py:127] - Time cost: 42.954s | Loading data Done
[3669930:MainThread](2025-09-01 17:06:55,986) INFO - qlib.timer - [log.py:127] - Time cost: 43.989s | RobustZScoreNorm Done
[3669930:MainThread](2025-09-01 17:07:04,461) INFO - qlib.timer - [log.py:127] - Time cost: 8.470s | Fillna Done
[3669930:MainThread](2025-09-01 17:07:13,065) INFO - qlib.timer - [log.py:127] - Time cost: 2.745s | DropnaLabel Done
[3669930:MainThread](2025-09-01 17:07:15,651) INFO - qlib.timer - [log.py:127] - Time cost: 2.583s | CSZScoreNorm Done
[3669930:MainThread](2025-09-01 17:07:15,666) INFO - qlib.timer - [log.py:127] - Time cost: 69.100s | fit & process data Done
[3669930:MainThread](2025-09-01 17:07:15,668) INFO - qlib.timer - [log.py:127] - Time cost: 112.058s | Init data Done
[3669930:MainThread](2025-09-01 17:08:46,786) INFO - qlib.timer - [log.py:127] - Time cost: 60.292s | Loading data Done
[3669930:MainThread](2025-09-01 17:09:48,6

In [3]:
last_day_scores = get_pred_scores_mix()

ModuleNotFoundError. CatBoostModel are skipped. (optional: maybe installing CatBoostModel can fix it.)
ModuleNotFoundError. XGBModel is skipped(optional: maybe installing xgboost can fix it).


In [4]:
stock_mix_compay = [
    "603235.SH", "600933.SH", "601958.SH", "600258.SH", "603868.SH",
    "601598.SH", "002242.SZ", "688009.SH", "601811.SH", "688389.SH",
    "601108.SH", "605338.SH", "603508.SH", "603708.SH", "601658.SH",
    "601688.SH", "601288.SH", "300682.SZ", "002511.SZ", "000959.SZ",
    "603368.SH", "002056.SZ", "601990.SH", "300606.SZ", "600061.SH",
    "300627.SZ", "002233.SZ", "688378.SH", "600999.SH", "002960.SZ",
    "688352.SH", "600596.SH", "002045.SZ", "002438.SZ", "002390.SZ",
    "603180.SH", "603408.SH", "301102.SZ", "601377.SH", "603309.SH",
    "300193.SZ", "002655.SZ", "600346.SH", "000404.SZ", "300999.SZ",
    "600109.SH", "603317.SH", "603816.SH", "600452.SH", "002673.SZ"
]
valid_codes = [code for code in stock_mix_compay if code in last_day_scores['instrument'].values]
sorted_df = last_day_scores[last_day_scores['instrument'].isin(valid_codes)].sort_values(
    by='score',
    ascending=False
).reset_index(drop=True)
sorted_df

,instrument,score
0,300682.SZ,0.469154
1,601958.SH,0.438707
2,601688.SH,0.398558
3,600061.SH,0.395196
4,600346.SH,0.389598
5,300999.SZ,0.156158
6,601108.SH,0.154008
7,601990.SH,0.148404
8,601377.SH,0.138622
9,688378.SH,0.120117


In [7]:
sorted_df[45:]['instrument'].tolist()

['300627.SZ', '600452.SH', '002390.SZ', '002045.SZ', '688389.SH']

In [8]:
top_100 = last_day_scores.sort_values('score', ascending=False).head(100)

# 重置索引（可选，如果希望 instrument 变成列）
top_100 = top_100.reset_index()
filtered_top_100 = top_100[~top_100['instrument'].isin(sorted_df['instrument'])]
filtered_top_100

,index,instrument,score
5,63,000776.SZ,0.354944
6,1272,688127.SH,0.349493
7,1382,688627.SH,0.313181
8,20,000408.SZ,0.273690
9,549,300790.SZ,0.261138
...,...,...,...
95,965,601801.SH,0.074475
96,744,600428.SH,0.074337
97,1118,603588.SH,0.074234
98,932,601456.SH,0.073885


In [10]:
filtered_top_100['instrument'][:10].tolist()

['000776.SZ',
 '688127.SH',
 '688627.SH',
 '000408.SZ',
 '300790.SZ',
 '600301.SH',
 '000938.SZ',
 '000676.SZ',
 '000977.SZ',
 '600895.SH']

In [16]:
# last_day_scores = get_pred_scores("gru_company", True)
last_day_scores = get_pred_scores("moe_company", True)

In [17]:
stock_codes_xqb01_online = [
    "603566", "603309", "603355", "603368", "601688", "688009",
    "601288", "600061", "601658", "688352", "601168", "600219",
    "601899", "002441", "600597", "603357", "603816", "600299",
    "603866", "601811", "001289", "601333", "001215", "002242",
    "605368", "300219", "688420", "600681", "603043", "000728",
    "601827", "601878", "601801", "002675", "688366", "601598",
    "603515", "603588", "300999", "002088", "000166", "601066",
    "603408", "600004", "601108", "000404", "002344", "603235",
    "002957", "688533"
]
# valid_codes = [code for code in stock_codes_xqb01_online if code in last_day_scores['instrument'].values]
valid_codes = [
    code for code in stock_codes_xqb01_online
    if code in [x[:6] for x in last_day_scores['instrument'].astype(str)]
]
sorted_df = last_day_scores[last_day_scores['instrument'].astype(str).str[:6].isin(valid_codes)].sort_values(
    by='score',
    ascending=True
).reset_index(drop=True)
sorted_df

,instrument,score
0,002344.SZ,0.015808
1,002675.SZ,0.027188
2,603309.SH,0.047715
3,600219.SH,0.047846
4,601168.SH,0.059815
5,601658.SH,0.061172
6,601899.SH,0.065650
7,603566.SH,0.065983
8,605368.SH,0.066257
9,603235.SH,0.068586


In [18]:
sorted_df[45:]['instrument'].tolist()

['603043.SH', '601688.SH', '600061.SH', '601066.SH', '000728.SZ']

In [19]:
top_100 = last_day_scores.sort_values('score', ascending=False).head(100)

# 重置索引（可选，如果希望 instrument 变成列）
top_100 = top_100.reset_index()
filtered_top_100 = top_100[~top_100['instrument'].isin(sorted_df['instrument'])]
filtered_top_100

,index,instrument,score
10,1007,601990.SH,0.191742
17,922,601319.SH,0.169688
18,598,301039.SZ,0.169614
19,927,601377.SH,0.168193
20,726,600339.SH,0.167406
...,...,...,...
95,548,300788.SZ,0.111579
96,282,002643.SZ,0.110890
97,938,601601.SH,0.110325
98,116,001872.SZ,0.109855


In [20]:
filtered_top_100['instrument'][:10].tolist()

['601990.SH',
 '601319.SH',
 '301039.SZ',
 '601377.SH',
 '600339.SH',
 '001323.SZ',
 '603730.SH',
 '688006.SH',
 '002233.SZ',
 '601019.SH']

In [4]:
folder_path='/home/quant/data_test/csv_data/daily_data'
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

In [2]:
section_code=pd.read_csv("section_codes.csv")

In [3]:
section_code

,000001
0,9
1,21
2,26
3,27
4,28
...,...
1404,688789
1405,688798
1406,688800
1407,688819


In [9]:
code_mapping={}
for code in csv_files:
    if len(code)>14 or code[:1]=='T':
        continue
    first_six=int(code[:6])
    first_nine=code[:9]
    code_mapping[first_six]=first_nine

In [10]:
code_mapping

{603121: '603121.SH',
 600006: '600006.SH',
 600055: '600055.SH',
 603007: '603007.SH',
 873690: '873690.BJ',
 2940: '002940.SZ',
 603466: '603466.SH',
 600189: '600189.SH',
 2658: '002658.SZ',
 688608: '688608.SH',
 505: '000505.SZ',
 301300: '301300.SZ',
 2706: '002706.SZ',
 301348: '301348.SZ',
 600098: '600098.SH',
 2559: '002559.SZ',
 301658: '301658.SZ',
 2250: '002250.SZ',
 301596: '301596.SZ',
 688721: '688721.SH',
 600928: '600928.SH',
 2082: '002082.SZ',
 301479: '301479.SZ',
 300319: '300319.SZ',
 939: '000939.SZ',
 600697: '600697.SH',
 603012: '603012.SH',
 2768: '002768.SZ',
 688235: '688235.SH',
 603239: '603239.SH',
 2324: '002324.SZ',
 2986: '002986.SZ',
 2112: '002112.SZ',
 600251: '600251.SH',
 839792: '839792.BJ',
 833030: '833030.BJ',
 603068: '603068.SH',
 603816: '603816.SH',
 688599: '688599.SH',
 603386: '603386.SH',
 301038: '301038.SZ',
 603877: '603877.SH',
 300997: '300997.SZ',
 1289: '001289.SZ',
 301112: '301112.SZ',
 300514: '300514.SZ',
 600568: '600568

In [11]:
section_code['stockcode']=section_code['000001'].map(code_mapping)

In [13]:
section_code['stockcode'].to_csv("/home/quant/zc/QuantBacktester_60/input/section_codes1.csv",index=False)